In [5]:
import os
import gc
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import cv2

# ==========================================
# 1. SETUP PATHS & EXTRACT FRAMES
# ==========================================
# Flexible search root: scans ./data_sample and all subfolders
target_video_dir = r"C:\Users\badet\Desktop\year 3\Software Engineer\project\wbesite_for_ai_checking\data_sample\data_sample\videos"
extracted_frames_dir = "./video_frames"

if os.path.exists(extracted_frames_dir):
    shutil.rmtree(extracted_frames_dir)

def extract_keyframes(video_path, output_dir, frame_interval=15):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_count, saved_count = 0, 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            frame_filename = os.path.join(output_dir, f"{video_name}_frame{saved_count:04d}.jpg")
            cv2.imwrite(frame_filename, frame)
            saved_count += 1
        frame_count += 1

    cap.release()
    return saved_count

real_count, fake_count = 0, 0
target_videos = []

valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')

for root, dirs, files in os.walk(target_video_dir):
    for file in files:
        if file.lower().endswith(valid_extensions):
            full_path = os.path.join(root, file)
            target_videos.append(full_path)
            
            path_and_name_lower = full_path.lower()
            if 'fake' in path_and_name_lower:
                dst = os.path.join(extracted_frames_dir, 'fake')
                fake_count += extract_keyframes(full_path, dst, frame_interval=15)
            elif 'real' in path_and_name_lower or 'photos' in path_and_name_lower:
                dst = os.path.join(extracted_frames_dir, 'real')
                real_count += extract_keyframes(full_path, dst, frame_interval=15)

print(f"Found {len(target_videos)} videos in '{target_video_dir}'.")
print(f"Extracted {real_count} REAL frames and {fake_count} FAKE frames.")

if len(target_videos) == 0:
    raise RuntimeError(f"No video files found in '{target_video_dir}'. Check your folder name and location!")

# ==========================================
# 2. GROUP-LEVEL SPLIT (PREVENT DATA LEAKAGE)
# ==========================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class_to_idx = {'fake': 0, 'real': 1}
video_to_frames = {}

for category in ['fake', 'real']:
    folder = os.path.join(extracted_frames_dir, category)
    if not os.path.exists(folder):
        continue
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg', '.png')):
            fpath = os.path.join(folder, fname)
            video_id = fname.split('_frame')[0]
            label = class_to_idx[category]
            
            if video_id not in video_to_frames:
                video_to_frames[video_id] = []
            video_to_frames[video_id].append((fpath, label))

unique_video_ids = list(video_to_frames.keys())

random.seed(42)
random.shuffle(unique_video_ids)

num_videos = len(unique_video_ids)
# Ensures at least 1 video goes to train even with small sample sizes
train_end = max(1, int(0.70 * num_videos))
val_end = train_end + max(1, int(0.15 * num_videos)) if num_videos > 2 else train_end

train_vids = unique_video_ids[:train_end]
val_vids = unique_video_ids[train_end:val_end] if num_videos > 1 else train_vids
test_vids = unique_video_ids[val_end:] if num_videos > 2 else val_vids

def gather_dataset_lists(vid_list):
    paths, labels = [], []
    for vid_id in vid_list:
        for fpath, lbl in video_to_frames[vid_id]:
            paths.append(fpath)
            labels.append(lbl)
    return paths, labels

train_paths, train_labels = gather_dataset_lists(train_vids)
val_paths, val_labels = gather_dataset_lists(val_vids)
test_paths, test_labels = gather_dataset_lists(test_vids)

print(f"\nGroup Split Summary -> Train Vids: {len(train_vids)} ({len(train_paths)} frames) | "
      f"Val Vids: {len(val_vids)} ({len(val_paths)} frames) | "
      f"Test Vids: {len(test_vids)} ({len(test_paths)} frames)")

# ==========================================
# 3. PYTORCH DATASET & TRANSFORMATIONS
# ==========================================
class VideoFrameDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = VideoFrameDataset(train_paths, train_labels, train_transform)
val_ds = VideoFrameDataset(val_paths, val_labels, val_test_transform)

train_loader = DataLoader(train_ds, batch_size=min(8, len(train_ds)), shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=min(8, len(val_ds)), shuffle=False, num_workers=0)

# ==========================================
# 4. CUSTOM CNN ARCHITECTURE
# ==========================================
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        self.fc1 = nn.Linear(256 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# ==========================================
# 5. MODEL TRAINING LOOP
# ==========================================
print(f"\nTraining CustomCNN on {device}...")
for epoch in range(15):
    model.train()
    running_loss, correct_train = 0.0, 0
    for inputs, lbls in train_loader:
        inputs, lbls = inputs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == lbls).sum().item()
        
    epoch_acc = (correct_train / len(train_ds)) * 100 if len(train_ds) > 0 else 0

    model.eval()
    correct_val = 0
    with torch.no_grad():
        for inputs, lbls in val_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == lbls).sum().item()
            
    val_acc = (correct_val / len(val_ds)) * 100 if len(val_ds) > 0 else 0
    print(f"Epoch {epoch+1:02d}/15 | Train Frame Acc: {epoch_acc:.2f}% | Val Frame Acc (Unseen Videos): {val_acc:.2f}%")

# ==========================================
# 6. EVALUATE HOLDOUT UNSEEN TEST VIDEOS
# ==========================================
print("\n" + "="*60)
print("🎬 EVALUATING HOLDOUT UNSEEN TEST VIDEOS")
print("="*60)

test_video_paths = [v for v in target_videos if os.path.splitext(os.path.basename(v))[0] in test_vids]

correct_count = 0
for vid_path in test_video_paths:
    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        continue
    
    frame_predictions = []
    frame_count = 0
    
    model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % 15 == 0:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                input_tensor = val_test_transform(img).unsqueeze(0).to(device)
                
                outputs = model(input_tensor)
                probs = F.softmax(outputs, dim=1)
                real_prob = probs[0][1].item()
                frame_predictions.append(real_prob)
                
            frame_count += 1
            
    cap.release()
    if not frame_predictions:
        continue
    
    avg_real_prob = np.mean(frame_predictions)
    avg_fake_prob = 1.0 - avg_real_prob
    
    verdict = "REAL" if avg_real_prob > 0.5 else "FAKE"
    confidence = max(avg_real_prob, avg_fake_prob) * 100
    
    vid_path_lower = vid_path.lower()
    actual_label = "FAKE" if 'fake' in vid_path_lower else "REAL"
    
    is_correct = "✅" if verdict == actual_label else "❌"
    if is_correct == "✅":
        correct_count += 1
    
    vid_name = os.path.basename(vid_path)
    print(f"[{is_correct}] Video: {vid_name:<30} | Pred: {verdict:<4} ({confidence:.1f}%) | Actual: {actual_label}")

total_test_vids = len(test_video_paths)
unseen_acc = (correct_count / total_test_vids) * 100 if total_test_vids > 0 else 0

print("="*60)
print(f"🏆 UNSEEN VIDEO TEST ACCURACY: {correct_count}/{total_test_vids} ({unseen_acc:.2f}%)")
print("="*60)

Found 20 videos in 'C:\Users\badet\Desktop\year 3\Software Engineer\project\wbesite_for_ai_checking\data_sample\data_sample\videos'.
Extracted 226 REAL frames and 124 FAKE frames.

Group Split Summary -> Train Vids: 14 (275 frames) | Val Vids: 3 (36 frames) | Test Vids: 3 (39 frames)

Training CustomCNN on cpu...
Epoch 01/15 | Train Frame Acc: 80.36% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 02/15 | Train Frame Acc: 93.09% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 03/15 | Train Frame Acc: 97.82% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 04/15 | Train Frame Acc: 100.00% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 05/15 | Train Frame Acc: 99.64% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 06/15 | Train Frame Acc: 100.00% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 07/15 | Train Frame Acc: 100.00% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 08/15 | Train Frame Acc: 100.00% | Val Frame Acc (Unseen Videos): 38.89%
Epoch 09/15 | Train Frame Acc: 100.00% | Val Frame Acc (U

In [6]:
torch.save(model.state_dict(), "video_cnn_weights.pth")
print("Video weights saved successfully!")

Video weights saved successfully!
